Eu tenho 100k newick na pasta mutation_dataset_100k_phylCNN

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ete3 import Tree
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split

c:\Users\JPC\anaconda3\envs\phylodeepenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\JPC\anaconda3\envs\phylodeepenv\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
TREE_DIR = r"mutation_dataset_100k_phyloCNN\\split_trees_corrected"
PARAM_FILE = r"mutation_dataset_100k_phyloCNN\\tabela_100k_phyloCNN.txt"

# BAD_INDICES = {
#    31035, 36174, 49680, 52494,
#    53227, 73155, 81488, 97073
#}

# no .nwk, a árvore dos ids de erro ta uma linha vazia, isso da problema
# acho melhor logo transformar o arquivo removendo as linhas vazias, e removendo os ids da tabela de parametros
# depois so precisa cuidar do v9, q vai 9992 arvores, mas qualquer coisa nbem treina dai

TARGET_COLUMNS = ["R_nought_1", "R_nought_2", "infectious_time"]

In [3]:
MUTANTS_FILE = r"mutation_dataset_100k_phyloCNN\\number_of_mutantes.npy"

#BAD_INDICES = [31035, 36174, 49680, 52494, 53227, 73155, 81488, 97073]
# Load parameter table
params = pd.read_csv(PARAM_FILE, sep="\t")

# Load number of mutants for each original tree
number_of_mutantes = np.load(MUTANTS_FILE)

# Check alignment before removing bad trees
if len(params) != len(number_of_mutantes):
    raise ValueError(
        f"Parameter table has {len(params)} rows, "
        f"but number_of_mutantes has {len(number_of_mutantes)} entries."
    )

# Add number of mutants
params["number_of_mutantes"] = number_of_mutantes


# Reset dataframe index
params.reset_index(drop=True, inplace=True)

# Total number of nodes in a full binary tree:
# total_nodes = 2 * internal_nodes + 1
params["total_nodes"] = 2 * params["tree_size"] + 1

# Fraction of mutated nodes
params["mutation_fraction"] = (
    params["number_of_mutantes"] / params["total_nodes"]
)


# <= 5 mutated nodes -> R02 is not reliably estimable
few_mutations = params["number_of_mutantes"] <= 5

params.loc[few_mutations, "R_nought_2"] = (
    params.loc[few_mutations, "R_nought_1"]
)

# Case 2:
# >95% of nodes mutated -> R01 is not reliably estimable
mostly_mutated = params["mutation_fraction"] > 0.95

params.loc[mostly_mutated, "R_nought_1"] = (
    params.loc[mostly_mutated, "R_nought_2"]
)

print("Original number of parameter rows:", len(number_of_mutantes))
#print("Removed bad trees:", len(BAD_INDICES))
print("Final parameter rows:", len(params))

print("Trees with <= 5 mutations:", few_mutations.sum())
print("Trees with >95% mutated nodes:", mostly_mutated.sum())

#print("\nBad indices still present:",
 #     sorted(set(params["index"]) & BAD_INDICES))


Original number of parameter rows: 99992
Final parameter rows: 99992
Trees with <= 5 mutations: 25985
Trees with >95% mutated nodes: 4962


In [4]:
params_by_id = params.set_index("index")

In [5]:
def ete_to_pyg(tree):
    nodes = list(tree.traverse())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    node_features = []
    edges = []
    edge_features = []

    mutation_root = -1

    for node in nodes:
        node_type = getattr(node, "ITYPE", None)
        parent_type = getattr(node.up, "ITYPE", None) if node.up is not None else None

        if mutation_root == -1 and node_type == 2 and parent_type != 2:
            mutation_root = node_to_idx[node]

        depth = node.get_distance(tree, topology_only=True)
        descendant_leaves = len(node.get_leaves())

        node_features.append([
            float(node.DIST_TO_START),
            float(node.is_leaf()),
            float(depth),
            float(descendant_leaves)
        ])

        if node.up is not None:
            parent = node_to_idx[node.up]
            child = node_to_idx[node]

            edges.append([parent, child])
            edges.append([child, parent])

            branch_length = float(node.dist)

            edge_features.append([branch_length])
            edge_features.append([branch_length])

    x = torch.tensor(node_features, dtype=torch.float32)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_features, dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, mutation_root=torch.tensor(mutation_root, dtype=torch.long))

In [6]:
def process_nwk(part_number, params):
    filename = os.path.join(TREE_DIR, f"trees_part_{part_number}.nwk")

    start_position = part_number * 10000
    end_position = start_position + 10000

    # Select the parameters corresponding to the trees in this file
    part_params = params.iloc[start_position:end_position].reset_index(drop=True)

    dataset = []

    with open(filename, "r") as f:
        tree_lines = [line.strip().replace(":i_t=", ":ITYPE=") for line in f if line.strip()]

    if len(tree_lines) != len(part_params):
        raise ValueError(
            f"Part {part_number}: "
            f"{len(tree_lines)} trees but "
            f"{len(part_params)} parameter rows."
        )

    for local_index, line in enumerate(tree_lines):

        # Original tree ID
        global_index = int(part_params.loc[local_index, "index"])

        #print(f"Processing tree {global_index} from part {part_number}")

        tree = Tree(line, format=1)


        graph = ete_to_pyg(tree)
        graph.tree_id = global_index

        dataset.append(graph)

    print(f"Part {part_number}: {len(dataset)} valid trees")

    return dataset

In [7]:
dataset = process_nwk(0, params)

Part 0: 10000 valid trees


In [8]:
invalid_roots = [i for i, graph in enumerate(dataset) 
                 if graph.mutation_root.item() < 0]

print("Trees without mutation root:", len(invalid_roots))

#we want 0

Trees without mutation root: 10000


In [9]:
def split_dataset(dataset):
    indices = np.arange(len(dataset))

    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    train_dataset = [dataset[i] for i in train_idx]
    val_dataset = [dataset[i] for i in val_idx]
    test_dataset = [dataset[i] for i in test_idx]

    return train_dataset, val_dataset, test_dataset

In [10]:
train_dataset, val_dataset, test_dataset = split_dataset(dataset)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

8000
1000
1000


In [11]:
node_features = torch.cat([g.x for g in train_dataset], dim=0)

continuous_features = node_features[:, [0, 2, 3]]

node_mean = continuous_features.mean(dim=0)
node_std = continuous_features.std(dim=0).clamp_min(1e-8)

In [12]:
def normalize_node_features(dataset, node_mean, node_std):
    for graph in dataset:
        graph.x[:, [0, 2, 3]] = (graph.x[:, [0, 2, 3]] - node_mean)/node_std

    return dataset

In [13]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Descomentar no treino seguitne
#checkpoint = torch.load("modelos\\GNN3\\v1.pt", map_location=device, weights_only=False)

#target_mean = torch.tensor(checkpoint["target_mean"], dtype=torch.float32)

#target_std = torch.tensor(checkpoint["target_std"], dtype=torch.float32)



In [14]:
train_dataset = normalize_node_features(train_dataset, node_mean, node_std)
val_dataset = normalize_node_features(val_dataset, node_mean, node_std)
test_dataset = normalize_node_features(test_dataset, node_mean, node_std)

In [15]:
class PhyloGNN(nn.Module):
    def __init__(self, num_features=4):
        super().__init__()

        self.gnn1 = GATConv(num_features, 64, heads=4, concat=False, edge_dim=1)
        self.gnn2 = GATConv(64, 64, heads=4, concat=False, edge_dim=1)
        self.gnn3 = GATConv(64, 64, heads=4, concat=False, edge_dim=1)

        self.root_head = nn.Linear(64, 1)
        self.no_mutation_head = nn.Linear(64, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        x = torch.relu(self.gnn1(x, edge_index, edge_attr))
        x = torch.relu(self.gnn2(x, edge_index, edge_attr))
        x = torch.relu(self.gnn3(x, edge_index, edge_attr))

        root_logits = self.root_head(x).squeeze(-1)

        graph_embedding = global_mean_pool(x, batch)
        no_mutation_logits = self.no_mutation_head(graph_embedding).squeeze(-1)

        return root_logits, no_mutation_logits

In [16]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
model = PhyloGNN(num_features=4).to(device)

#descomentar no treino seguinte
#!!!!!
#model.load_state_dict(checkpoint["model_state_dict"])

model.train()

PhyloGNN(
  (gnn1): GATConv(4, 64, heads=4)
  (gnn2): GATConv(64, 64, heads=4)
  (gnn3): GATConv(64, 64, heads=4)
  (root_head): Linear(in_features=64, out_features=1, bias=True)
  (no_mutation_head): Linear(in_features=64, out_features=1, bias=True)
)

In [18]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        root_logits, no_mutation_logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        losses = []

        for graph_idx in range(batch.num_graphs):

            node_mask = batch.batch == graph_idx

            graph_logits = root_logits[node_mask]
            logits = torch.cat([graph_logits, no_mutation_logits[graph_idx].unsqueeze(0)])
            target = batch.mutation_root[graph_idx]

            if target >= 0:
                target_class = target
            else:
                target_class = torch.tensor(graph_logits.shape[0], device=device)

            loss = criterion(logits.unsqueeze(0), target_class.unsqueeze(0))

            losses.append(loss)

        loss = torch.stack(losses).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [19]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        root_logits, no_mutation_logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        losses = []

        for graph_idx in range(batch.num_graphs):

            node_mask = batch.batch == graph_idx

            graph_logits = root_logits[node_mask]
            logits = torch.cat([graph_logits, no_mutation_logits[graph_idx].unsqueeze(0)])
            target = batch.mutation_root[graph_idx]

            if target >= 0:
                target_class = target
            else:
                target_class = torch.tensor(graph_logits.shape[0], device=device)

            loss = criterion(logits.unsqueeze(0), target_class.unsqueeze(0))

            losses.append(loss)

        loss = torch.stack(losses).mean()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [20]:
@torch.no_grad()
def root_accuracy(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        root_logits, no_mutation_logits = model(
            batch.x, batch.edge_index, batch.edge_attr, batch.batch
        )

        for graph_idx in range(batch.num_graphs):
            node_mask = batch.batch == graph_idx

            graph_logits = root_logits[node_mask]

            logits = torch.cat([
                graph_logits,
                no_mutation_logits[graph_idx].unsqueeze(0)
            ])

            predicted_class = torch.argmax(logits)
            true_root = batch.mutation_root[graph_idx]

            if true_root >= 0:
                correct += (predicted_class == true_root).item()
            else:
                no_mutation_class = graph_logits.shape[0]
                correct += (predicted_class == no_mutation_class).item()

            total += 1

    return correct / total

In [21]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [22]:
graph = dataset[0]

print(graph)
print("Node features:", graph.x.shape)
print("Edges:", graph.edge_index.shape)
print("Edge features:", graph.edge_attr.shape)
print(graph.edge_attr[:10])

Data(x=[907, 4], edge_index=[2, 1812], edge_attr=[1812, 1], mutation_root=-1, tree_id=0)
Node features: torch.Size([907, 4])
Edges: torch.Size([2, 1812])
Edge features: torch.Size([1812, 1])
tensor([[3.1361],
        [3.1361],
        [3.0498],
        [3.0498],
        [4.4028],
        [4.4028],
        [0.2010],
        [0.2010],
        [3.6532],
        [3.6532]])


In [24]:
best_val_loss = float("inf")

for epoch in range(1, 10):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    train_acc = root_accuracy(model, train_loader, device)
    val_acc = root_accuracy(model, val_loader, device)
    print(f"Epoch {epoch:03d} | Train: {train_loss:.5f} | Val: {val_loss:.5f}"
          f"| Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

        torch.save({
                "model_state_dict": model.state_dict(),
                "node_mean": node_mean.cpu().numpy(),
                "node_std": node_std.cpu().numpy(),
                "best_val_loss": best_val_loss,
                "epoch": best_epoch}, "modelos\\GNN3-mut-node\\v0.pt")

Epoch 001 | Train: 0.00001 | Val: 0.00001| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 002 | Train: 0.00001 | Val: 0.00001| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 003 | Train: 0.00001 | Val: 0.00001| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 004 | Train: 0.00001 | Val: 0.00001| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 005 | Train: 0.00001 | Val: 0.00001| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 006 | Train: 0.00000 | Val: 0.00000| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 007 | Train: 0.00000 | Val: 0.00000| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 008 | Train: 0.00000 | Val: 0.00000| Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 009 | Train: 0.00000 | Val: 0.00000| Train Acc: 1.0000 | Val Acc: 1.0000


In [ ]:
best_checkpoint = torch.load("modelos\\GNN3-mut-node\\v0.pt", map_location=device, weights_only=False)

model.load_state_dict(best_checkpoint["model_state_dict"])

model.eval()

print("Best epoch:",best_checkpoint["epoch"])

print("Best validation loss:", best_checkpoint["best_val_loss"])

test_acc = root_accuracy(model, test_loader, device)

print(f"Test root accuracy: {test_acc:.4f}")
print(f"Test root accuracy: {test_acc * 100:.2f}%")


Best epoch: 97
Best validation loss: 0.14551273858547212


In [ ]:
@torch.no_grad()
def root_distance(model, loader, device):
    model.eval()

    distances = []

    for batch in loader:

        batch = batch.to(device)

        root_logits = model(batch.x, batch.edge_index, batch.edge_attr)

        for graph_idx in range(batch.num_graphs):

            node_mask = batch.batch == graph_idx

            graph_logits = root_logits[node_mask]

            predicted_root = torch.argmax(graph_logits)

            true_root = batch.mutation_root[graph_idx]

            distances.append(abs(predicted_root.item() - true_root.item()))

    return np.mean(distances), np.median(distances)

In [ ]:
mean_dist, median_dist = root_distance(model, test_loader, device)

print(f"Mean root index distance: {mean_dist:.2f}")
print(f"Median root index distance: {median_dist:.2f}")

In [ ]:
test_acc = root_accuracy(model, test_loader, device)

print(f"Test root accuracy: {test_acc:.4f}")
print(f"Test root accuracy: {test_acc * 100:.2f}%")

In [ ]:
n_positions = np.load(r"C:\Users\JPC\Documents\MESTRADO\Projeto\gillespie-2\mutation_dataset_100k_phyloCNN\mutation_positions.npy")
print(len(n_positions)) 

99992
